In [33]:
import pandas as pd
import numpy as np

In [34]:
monthly_df = pd.read_csv(r"C:\Users\naksh\Projects\xgboost_stock_ranking\data\raw\nifty100_monthly.csv")

In [35]:
monthly_df["mom_1m"] = (
    monthly_df
    .groupby("Ticker")["Close"]
    .pct_change(1)
)

In [36]:
monthly_df["mom_3m"] = (
    monthly_df
    .groupby("Ticker")["Close"]
    .pct_change(3)
)

In [37]:
monthly_df["mom_6m"] = (
    monthly_df
    .groupby("Ticker")["Close"]
    .pct_change(6)
)


In [38]:
monthly_df["mom_12m"] = (
    monthly_df
    .groupby("Ticker")["Close"]
    .pct_change(12)
)


In [39]:
monthly_df["ret_1m"] = (
    monthly_df
    .groupby("Ticker")["Close"]
    .pct_change()
)

In [40]:
monthly_df["vol_3m"] = (
    monthly_df
    .groupby("Ticker")["ret_1m"]
    .rolling(3)
    .std()
    .reset_index(level=0, drop=True)
)

monthly_df["vol_6m"] = (
    monthly_df
    .groupby("Ticker")["ret_1m"]
    .rolling(6)
    .std()
    .reset_index(level=0, drop=True)
)

monthly_df["vol_12m"] = (
    monthly_df
    .groupby("Ticker")["ret_1m"]
    .rolling(12)
    .std()
    .reset_index(level=0, drop=True)
)

In [41]:
monthly_df["avg_vol_3m"] = (
    monthly_df
    .groupby("Ticker")["Volume"]
    .rolling(3)
    .mean()
    .reset_index(level=0, drop=True)
)

monthly_df["rel_volume"] = (
    monthly_df["Volume"]
    / monthly_df["avg_vol_3m"]
)

monthly_df["vol_growth"] = (
    monthly_df
    .groupby("Ticker")["Volume"]
    .pct_change(3)
)

In [42]:

monthly_df["ma_12"] = (
    monthly_df
    .groupby("Ticker")["Close"]
    .rolling(12)
    .mean()
    .reset_index(level=0, drop=True)
)

monthly_df["price_ma_ratio"] = (
    monthly_df["Close"]
    / monthly_df["ma_12"]
)

In [43]:
monthly_df["skew_12m"] = (
    monthly_df
    .groupby("Ticker")["ret_1m"]
    .rolling(12)
    .skew()
    .reset_index(level=0, drop=True)
)

monthly_df["max_return_12m"] = (
    monthly_df
    .groupby("Ticker")["ret_1m"]
    .rolling(12)
    .max()
    .reset_index(level=0, drop=True)
)

In [44]:
feature_cols = [
    "mom_1m",
    "mom_3m",
    "mom_6m",
    "mom_12m",
    "vol_3m",
    "vol_6m",
    "vol_12m",
    "price_ma_ratio",
    "rel_volume",
    "vol_growth",
    "skew_12m",
    "max_return_12m"
]

In [45]:
monthly_df[feature_cols].isnull().sum()

mom_1m              97
mom_3m             290
mom_6m             578
mom_12m           1154
vol_3m             290
vol_6m             578
vol_12m           1154
price_ma_ratio    1058
rel_volume         194
vol_growth         290
skew_12m          1154
max_return_12m    1154
dtype: int64

In [46]:
monthly_df = monthly_df.dropna(
    subset=feature_cols + ["target_return"]
)

In [47]:
monthly_df[feature_cols].isnull().sum()

mom_1m            0
mom_3m            0
mom_6m            0
mom_12m           0
vol_3m            0
vol_6m            0
vol_12m           0
price_ma_ratio    0
rel_volume        0
vol_growth        0
skew_12m          0
max_return_12m    0
dtype: int64

In [48]:
monthly_df.shape

(9536, 24)

In [49]:
monthly_df.head()

,Unnamed: 0,Ticker,Date,Open,High,Low,Close,Volume,target_return,mom_1m,...,vol_3m,vol_6m,vol_12m,avg_vol_3m,rel_volume,vol_growth,ma_12,price_ma_ratio,skew_12m,max_return_12m
12,12,ABB.NS,2016-01-31,840.381501,850.019082,835.258795,843.159912,20531.0,0.043456,-0.128668,...,0.044851,0.114468,0.097736,29047.333333,0.706812,0.244982,1072.711029,0.786008,0.498894,0.168247
13,13,ABB.NS,2016-02-29,892.563540,949.868171,870.813795,879.800232,98488.0,0.265321,0.043456,...,0.086115,0.115450,0.092991,47584.000000,2.069771,1.296935,1044.348760,0.842439,0.573702,0.168247
14,14,ABB.NS,2016-03-31,1080.974300,1125.949796,1068.862253,1113.229858,127940.0,0.014851,0.265321,...,0.197517,0.149887,0.120526,82319.666667,1.554185,4.390806,1046.226089,1.064043,0.844864,0.265321
15,15,ABB.NS,2016-04-30,1105.988418,1140.822682,1094.667198,1129.762817,62604.0,-0.046481,0.014851,...,0.137099,0.136307,0.119307,96344.000000,0.649797,2.049243,1043.718409,1.082440,0.982275,0.265321
16,16,ABB.NS,2016-05-31,1058.091380,1082.475378,1055.565873,1077.250244,44332.0,-0.010550,-0.046481,...,0.165185,0.135515,0.120199,78292.000000,0.566239,-0.549874,1036.816594,1.038998,1.069013,0.265321


In [50]:
monthly_df[feature_cols + ["target_return"]].describe()

c:\Users\naksh\Projects\xgboost_stock_ranking\.venv\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,mom_1m,mom_3m,mom_6m,mom_12m,vol_3m,vol_6m,vol_12m,price_ma_ratio,rel_volume,vol_growth,skew_12m,max_return_12m,target_return
count,9536.000000,9536.000000,9536.000000,9536.000000,9536.000000,9536.000000,9536.000000,9536.000000,9536.000000,9536.000000,9536.000000,9536.000000,9536.000000
mean,0.021845,0.068496,0.144404,0.310789,0.083868,0.090776,0.093792,1.101678,1.003149,inf,0.163469,0.190456,0.022211
std,0.104690,0.197226,0.327650,0.661731,0.061732,0.051047,0.045121,0.214349,0.477881,NaN,0.772339,0.119857,0.104274
min,-0.637599,-0.778958,-0.837680,-0.883641,0.000514,0.009806,0.019283,0.246402,0.010775,-0.992055,-2.654739,0.012697,-0.637599
25%,-0.038082,-0.045985,-0.042744,-0.024504,0.045158,0.059328,0.065683,0.978047,0.659328,-0.393518,-0.332934,0.116602,-0.037739
50%,0.016172,0.050154,0.100492,0.192567,0.070494,0.078748,0.082273,1.083891,0.938175,-0.016414,0.172699,0.161839,0.016180
75%,0.074132,0.160909,0.268513,0.482696,0.105611,0.107358,0.108245,1.207974,1.278927,0.656246,0.655113,0.230963,0.074168
max,1.408602,2.733333,5.866667,13.266667,0.768015,0.631200,0.491295,2.876388,2.959313,inf,2.965812,1.408602,1.408602


In [51]:
print(monthly_df.shape)

print(monthly_df["Ticker"].nunique())

print(monthly_df["Date"].min())
print(monthly_df["Date"].max())

(9536, 24)
96
2016-01-31
2024-11-30


In [52]:
np.isinf(monthly_df["vol_growth"]).sum()

np.int64(1)

In [53]:
monthly_df[
    np.isinf(monthly_df["vol_growth"])
][["Ticker", "Date", "Volume", "vol_growth"]].head(20)

,Ticker,Date,Volume,vol_growth
8226,SHRIRAMFIN.NS,2016-02-29,3836247.0,inf


In [54]:
monthly_df.head()

,Unnamed: 0,Ticker,Date,Open,High,Low,Close,Volume,target_return,mom_1m,...,vol_3m,vol_6m,vol_12m,avg_vol_3m,rel_volume,vol_growth,ma_12,price_ma_ratio,skew_12m,max_return_12m
12,12,ABB.NS,2016-01-31,840.381501,850.019082,835.258795,843.159912,20531.0,0.043456,-0.128668,...,0.044851,0.114468,0.097736,29047.333333,0.706812,0.244982,1072.711029,0.786008,0.498894,0.168247
13,13,ABB.NS,2016-02-29,892.563540,949.868171,870.813795,879.800232,98488.0,0.265321,0.043456,...,0.086115,0.115450,0.092991,47584.000000,2.069771,1.296935,1044.348760,0.842439,0.573702,0.168247
14,14,ABB.NS,2016-03-31,1080.974300,1125.949796,1068.862253,1113.229858,127940.0,0.014851,0.265321,...,0.197517,0.149887,0.120526,82319.666667,1.554185,4.390806,1046.226089,1.064043,0.844864,0.265321
15,15,ABB.NS,2016-04-30,1105.988418,1140.822682,1094.667198,1129.762817,62604.0,-0.046481,0.014851,...,0.137099,0.136307,0.119307,96344.000000,0.649797,2.049243,1043.718409,1.082440,0.982275,0.265321
16,16,ABB.NS,2016-05-31,1058.091380,1082.475378,1055.565873,1077.250244,44332.0,-0.010550,-0.046481,...,0.165185,0.135515,0.120199,78292.000000,0.566239,-0.549874,1036.816594,1.038998,1.069013,0.265321


In [55]:
monthly_df.columns

Index(['Unnamed: 0', 'Ticker', 'Date', 'Open', 'High', 'Low', 'Close',
       'Volume', 'target_return', 'mom_1m', 'mom_3m', 'mom_6m', 'mom_12m',
       'ret_1m', 'vol_3m', 'vol_6m', 'vol_12m', 'avg_vol_3m', 'rel_volume',
       'vol_growth', 'ma_12', 'price_ma_ratio', 'skew_12m', 'max_return_12m'],
      dtype='str')

In [56]:
monthly_df = monthly_df.drop(
    columns=["Unnamed: 0"],
    errors="ignore"
)

In [57]:
monthly_df = monthly_df.reset_index(drop=True)

In [58]:
monthly_df.head()

,Ticker,Date,Open,High,Low,Close,Volume,target_return,mom_1m,mom_3m,...,vol_3m,vol_6m,vol_12m,avg_vol_3m,rel_volume,vol_growth,ma_12,price_ma_ratio,skew_12m,max_return_12m
0,ABB.NS,2016-01-31,840.381501,850.019082,835.258795,843.159912,20531.0,0.043456,-0.128668,-0.215621,...,0.044851,0.114468,0.097736,29047.333333,0.706812,0.244982,1072.711029,0.786008,0.498894,0.168247
1,ABB.NS,2016-02-29,892.563540,949.868171,870.813795,879.800232,98488.0,0.265321,0.043456,-0.134302,...,0.086115,0.115450,0.092991,47584.000000,2.069771,1.296935,1044.348760,0.842439,0.573702,0.168247
2,ABB.NS,2016-03-31,1080.974300,1125.949796,1068.862253,1113.229858,127940.0,0.014851,0.265321,0.150426,...,0.197517,0.149887,0.120526,82319.666667,1.554185,4.390806,1046.226089,1.064043,0.844864,0.265321
3,ABB.NS,2016-04-30,1105.988418,1140.822682,1094.667198,1129.762817,62604.0,-0.046481,0.014851,0.339915,...,0.137099,0.136307,0.119307,96344.000000,0.649797,2.049243,1043.718409,1.082440,0.982275,0.265321
4,ABB.NS,2016-05-31,1058.091380,1082.475378,1055.565873,1077.250244,44332.0,-0.010550,-0.046481,0.224426,...,0.165185,0.135515,0.120199,78292.000000,0.566239,-0.549874,1036.816594,1.038998,1.069013,0.265321


In [59]:
monthly_df[
    np.isinf(monthly_df["vol_growth"])
][["Ticker", "Date", "Volume", "vol_growth"]].head(20)

,Ticker,Date,Volume,vol_growth
7312,SHRIRAMFIN.NS,2016-02-29,3836247.0,inf


In [60]:
monthly_df = monthly_df.drop(7312)

In [61]:
monthly_df[
    np.isinf(monthly_df["vol_growth"])
][["Ticker", "Date", "Volume", "vol_growth"]].head(20)

,Ticker,Date,Volume,vol_growth


In [62]:
monthly_df = monthly_df.drop(
    columns=[
        "ret_1m",
        "ma_12",
        "avg_vol_3m"
    ]
)

In [63]:
monthly_df.to_csv(
    "../data/processed/monthly_features.csv",
    index=False
)